# Stim ON vs Stim OFF — Trial Count & Accuracy Comparison
### Digit Span Backwards | DBS ATN Case 1 | Sessions 2 & 3

**Two Classification Ideas, Two Conditions Each:**

**Idea 1 — Trial Count (y-axis: number of trials)**
- Condition A: Any stimulation ≥ 2.0 mA anywhere in the trial → STIM-ON
- Condition B: Feedback-window-only stimulation → reclassified as NO-STIM

**Idea 2 — Accuracy (y-axis: proportion correct)**
- Same two conditions as Idea 1

Each idea plotted in a **2 × 3** grid (2 conditions × 3 comparisons: Session 2, Session 3, Combined)


## Cell 1 — Imports & Display Settings

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
from scipy import stats
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
plt.rcParams.update({
    'font.family':       'sans-serif',
    'font.size':         10,
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'figure.dpi':        130,
    'savefig.dpi':       160,
    'savefig.bbox':      'tight',
    'savefig.facecolor': 'white',
})

# ── Colours ──────────────────────────────────────────────────────────────────
C_NO_STIM  = '#90A4AE'   # grey  — no stimulation
C_STIM     = '#1A56DB'   # blue  — stimulation present
C_CORRECT  = '#2E7D32'   # green
C_WRONG    = '#C62828'   # red

STIM_THRESHOLD = 1     # mA — threshold that defines "stim on"

print('Imports OK')

## Cell 2 — File Paths  ← EDIT THESE

In [ ]:
JSON_PATH_S2   = Path(r"C:\Users\ASSUS\ATN\Digit Span Backwards\Data\Neural Data\DBS ATN DSB Case 1\D. Siragusa\3.5.26\Time stamp 1433\Report_Json_Session_Report_20260305T151703.json")
CSV_PATH_S2    = Path(r"C:\Users\ASSUS\ATN\Digit Span Backwards\Data\Eprime Data\Digit Span Backwards v3.2\DigitSpanBackward v3.3-6-2-Scores.csv")
EVENTS_PATH_S2 = Path(r"C:\Users\ASSUS\ATN\Digit Span Backwards\Session 2\Events.csv")
OUT_DIR_S2     = Path(r"C:\Users\ASSUS\ATN\Digit Span Backwards\Session 2")

JSON_PATH_S3   = Path(r"C:\Users\ASSUS\ATN\Digit Span Backwards\Data\Neural Data\DBS ATN DSB Case 1\D. Siragusa\3.5.26\Time stamp 1441\Report_Json_Session_Report_20260305T151912.json")
CSV_PATH_S3    = Path(r"C:\Users\ASSUS\ATN\Digit Span Backwards\Data\Eprime Data\Digit Span Backwards v3.2\DigitSpanBackward v3.3-6-3-Scores.csv")
EVENTS_PATH_S3 = Path(r"C:\Users\ASSUS\ATN\Digit Span Backwards\Session 3\Preprocessed Data\Events.csv")
OUT_DIR_S3     = Path(r"C:\Users\ASSUS\ATN\Digit Span Backwards\Session 3")

COMBINED_DIR = Path(r"C:\Users\ASSUS\ATN\Digit Span Backwards\Session 2 v 3\Base\Stim on vs Stim off\Visuals")
COMBINED_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR_S2.mkdir(parents=True, exist_ok=True)
OUT_DIR_S3.mkdir(parents=True, exist_ok=True)

print('Paths set.')

## Cell 3 — Data Loading & Trial Classification Functions

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Load raw session data (LFP JSON + E-Prime CSV + Events CSV)
# ─────────────────────────────────────────────────────────────────────────────
def load_session_data(json_path, csv_path, events_path):
    with open(json_path) as f:
        report = json.load(f)
    eprime_df = pd.read_csv(csv_path,    encoding='utf-8-sig', low_memory=False)
    ev_df     = pd.read_csv(events_path, encoding='utf-8-sig', low_memory=False)

    meta = dict(
        subject = str(eprime_df['Subject'].iloc[0]),
        session = str(eprime_df['Session'].iloc[0]),
        date    = str(eprime_df['SessionDate'].iloc[0]),
    )

    stim_tick = None
    for stream in report['BrainSenseLfp']:
        prev = None
        for pkt in stream['LfpData']:
            curr = pkt['Left']['mA']
            if prev is not None and prev == 0.0 and curr > 0.0:
                stim_tick = pkt['TicksInMs']; break
            prev = curr
        if stim_tick: break
    assert stim_tick, 'No 0->+mA transition found!'

    welcome_ms    = int(eprime_df['Welcome.TargetOnsetTime'].iloc[0])
    MANUAL_OFFSET = stim_tick - welcome_ms
    def to_rel(ms): return float(ms) + MANUAL_OFFSET - stim_tick

    ticks, mAs = [], []
    for stream in report['BrainSenseLfp']:
        for pkt in stream['LfpData']:
            ticks.append(pkt['TicksInMs'])
            mAs.append(pkt['Left']['mA'])
    ticks = np.array(ticks, dtype=float)
    mAs   = np.array(mAs,   dtype=float)
    order = np.argsort(ticks)
    ticks_rel = ticks[order] - stim_tick
    mAs       = mAs[order]

    print(f"Loaded: Subject={meta['subject']}  Session={meta['session']}  "
          f"Date={meta['date']}  mA_max={mAs.max():.2f}  N_packets={len(ticks)}")
    return report, eprime_df, ev_df, meta, to_rel, ticks_rel, mAs


def peak_mA(t0, t1, ticks_rel, mAs):
    if t0 is None or t1 is None or t1 <= t0: return 0.0
    mask = (ticks_rel >= t0) & (ticks_rel <= t1)
    if not mask.any():
        idx = int(np.argmin(np.abs(ticks_rel - (t0+t1)/2)))
        return float(mAs[idx])
    return float(mAs[mask].max())


def stim_frac_in_window(t0, t1, ticks_rel, mAs, threshold=2.0):
    if t0 is None or t1 is None or t1 <= t0: return 0.0
    mask = (ticks_rel >= t0) & (ticks_rel <= t1)
    t_r, t_m = ticks_rel[mask], mAs[mask]
    if len(t_r) < 2: return 0.0
    dt    = np.diff(t_r)
    mid   = (t_m[:-1] + t_m[1:]) / 2.0
    total = t_r[-1] - t_r[0]
    return float(np.sum(dt[mid >= threshold]) / total) if total > 0 else 0.0


def build_all_trials(eprime_df, ev_df, to_rel, ticks_rel, mAs, sess_label):
    digit_rows = eprime_df['Digit'].tolist()
    trial_digit_seqs = {}
    offset = 0
    for tn in range(1, 15):
        row = ev_df[(ev_df['Event_Type']=='Main Trial Start') & (ev_df['Trial_Number']==tn)]
        if row.empty: continue
        n = int(row.iloc[0]['Num_Digits'])
        trial_digit_seqs[tn] = digit_rows[offset:offset+n]
        offset += n

    def ev_all(etype, tn):
        return ev_df[(ev_df['Event_Type']==etype) &
                     (ev_df['Trial_Number']==tn)]['Time_ms'].tolist()
    def ev_first(etype, tn):
        v = ev_all(etype, tn); return v[0] if v else None

    trials = []
    for tn in range(1, 15):
        sr = ev_df[(ev_df['Event_Type']=='Main Trial Start') & (ev_df['Trial_Number']==tn)]
        if sr.empty: continue
        r = sr.iloc[0]

        acc     = int(r['ACC'])        if pd.notna(r['ACC'])        else None
        cresp_v = r['CRESP']           if pd.notna(r['CRESP'])      else None
        resp_v  = r['RESP']            if pd.notna(r['RESP'])       else None
        nd      = int(r['Num_Digits']) if pd.notna(r['Num_Digits']) else None
        if None in (acc, cresp_v, resp_v, nd): continue

        cresp_s   = str(int(cresp_v)).zfill(nd)
        resp_s    = str(int(resp_v)).zfill(nd)
        presented = cresp_s[::-1]

        t_start_ms = ev_first('Main Trial Start', tn)
        t_end_ms   = ev_first('Main Trial End',   tn)
        t_start    = to_rel(t_start_ms) if t_start_ms else None
        t_end      = to_rel(t_end_ms)   if t_end_ms   else None

        fix_starts  = [to_rel(ms) for ms in ev_all('Fixation Start', tn)]
        stim_starts = [to_rel(ms) for ms in ev_all('Stimulus Start', tn)]
        stim_ends   = [to_rel(ms) for ms in ev_all('Stimulus End',   tn)]
        cs_ms = ev_first('Choice Start',   tn); cs = to_rel(cs_ms) if cs_ms else None
        ce_ms = ev_first('Choice End',     tn); ce = to_rel(ce_ms) if ce_ms else None
        fs_ms = ev_first('Feedback Start', tn); fs = to_rel(fs_ms) if fs_ms else None
        fe_ms = ev_first('Feedback End',   tn); fe = to_rel(fe_ms) if fe_ms else None

        # ── Condition A: any stim anywhere in trial (fix -> feedback end)
        trial_lo = fix_starts[0] if fix_starts else t_start
        trial_hi = fe if fe else t_end

        # Check if ANY stimulation ≥ threshold exists in window
        mask = (ticks_rel >= trial_lo) & (ticks_rel <= trial_hi)

        if mask.any():
            stim_present_condA = np.any(mAs[mask] >= STIM_THRESHOLD)
        else:
            stim_present_condA = False

        # ── Condition B: stim ONLY in feedback window
        # Check if stim occurs in pre-feedback windows (fix, stim, choice)
        pre_fb_frac = 0.0
        if fix_starts and stim_starts:
            pre_fb_frac = max(pre_fb_frac,
                stim_frac_in_window(fix_starts[0], stim_starts[0], ticks_rel, mAs, STIM_THRESHOLD))
        if stim_starts and stim_ends:
            pre_fb_frac = max(pre_fb_frac,
                stim_frac_in_window(stim_starts[0], stim_ends[0], ticks_rel, mAs, STIM_THRESHOLD))
        if cs and ce:
            pre_fb_frac = max(pre_fb_frac,
                stim_frac_in_window(cs, ce, ticks_rel, mAs, STIM_THRESHOLD))
        fb_frac = stim_frac_in_window(fs, fe, ticks_rel, mAs, STIM_THRESHOLD) if fs and fe else 0.0

        # Condition B: if stim ONLY in feedback → treat as NO-STIM
        # i.e. stim_present for condB = stim present AND also present pre-feedback
        feedback_only_stim = stim_present_condA and (pre_fb_frac < 0.3) and (fb_frac >= 0.3)
        stim_present_condB = stim_present_condA and not feedback_only_stim

        # Per-window peak mA
        fix_peak  = max((peak_mA(fs2, ss, ticks_rel, mAs)
                         for fs2, ss in zip(fix_starts, stim_starts)), default=0.0)
        stim_peak = max((peak_mA(ss, se, ticks_rel, mAs)
                         for ss, se in zip(stim_starts, stim_ends)), default=0.0)
        ch_peak   = peak_mA(cs, ce, ticks_rel, mAs)
        fb_peak   = peak_mA(fs, fe, ticks_rel, mAs)
        whole_peak = peak_mA(trial_lo, trial_hi, ticks_rel, mAs)

        trials.append(dict(
            session              = sess_label,
            trial_num            = tn,
            digits               = nd,
            acc                  = acc,
            presented            = presented,
            cresp                = cresp_s,
            resp                 = resp_s,
            digit_seq            = trial_digit_seqs.get(tn, []),
            stim_present         = stim_present_condA,   # Condition A
            stim_present_condB   = stim_present_condB,   # Condition B
            feedback_only_stim   = feedback_only_stim,
            whole_peak_mA        = round(whole_peak, 3),
            fix_peak_mA          = round(fix_peak,  3),
            stim_peak_mA         = round(stim_peak, 3),
            ch_peak_mA           = round(ch_peak,   3),
            fb_peak_mA           = round(fb_peak,   3),
        ))

    return trials


print('Functions defined.')


## Cell 4 — Statistical Helper Functions

In [ ]:
def cohens_d(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    if len(a) < 2 or len(b) < 2: return np.nan
    pooled = np.sqrt(((len(a)-1)*np.var(a, ddof=1) +
                      (len(b)-1)*np.var(b, ddof=1)) / (len(a)+len(b)-2))
    return float((np.mean(a) - np.mean(b)) / pooled) if pooled > 0 else np.nan

def effect_label(d):
    if np.isnan(d): return 'N/A'
    v = abs(d)
    if v >= 0.8: return 'large'
    if v >= 0.5: return 'medium'
    if v >= 0.2: return 'small'
    return 'negligible'

def sig_stars(p):
    if np.isnan(p): return ''
    if p < 0.001: return '***'
    if p < 0.01:  return '**'
    if p < 0.05:  return '*'
    return 'ns'

def run_comparison(no_stim_trials, stim_trials, label):
    acc_no  = np.array([t['acc'] for t in no_stim_trials], dtype=float)
    acc_on  = np.array([t['acc'] for t in stim_trials],    dtype=float)

    n_no, n_corr_no = len(acc_no), int(acc_no.sum())
    n_on, n_corr_on = len(acc_on), int(acc_on.sum())

    pct_no = acc_no.mean()*100 if n_no > 0 else np.nan
    pct_on = acc_on.mean()*100 if n_on > 0 else np.nan

    # Fisher's exact (2x2: correct/wrong x no-stim/stim)
    if n_no > 0 and n_on > 0:
        _, p = stats.fisher_exact([
            [n_corr_no,  n_no - n_corr_no],
            [n_corr_on,  n_on - n_corr_on]
        ])
        p = float(p)
    else:
        p = np.nan

    d = cohens_d(acc_no, acc_on)

    return dict(
        label        = label,
        n_no_stim    = n_no,
        correct_no   = n_corr_no,
        pct_no       = pct_no,
        n_stim       = n_on,
        correct_on   = n_corr_on,
        pct_on       = pct_on,
        p_value      = p,
        stars        = sig_stars(p),
        cohens_d     = d,
        effect_size  = effect_label(d),
    )

print('Stats helpers defined.')

## Cell 5 — Load Sessions & Classify Trials

In [ ]:
# ── Session 2 ────────────────────────────────────────────────────────────────
_, ep_s2, ev_s2, meta_s2, to_rel_s2, tr_s2, mA_s2 = \
    load_session_data(JSON_PATH_S2, CSV_PATH_S2, EVENTS_PATH_S2)
all_s2 = build_all_trials(ep_s2, ev_s2, to_rel_s2, tr_s2, mA_s2, 'Session 2')

print()

# ── Session 3 ────────────────────────────────────────────────────────────────
_, ep_s3, ev_s3, meta_s3, to_rel_s3, tr_s3, mA_s3 = \
    load_session_data(JSON_PATH_S3, CSV_PATH_S3, EVENTS_PATH_S3)
all_s3 = build_all_trials(ep_s3, ev_s3, to_rel_s3, tr_s3, mA_s3, 'Session 3')

# ── Combined ─────────────────────────────────────────────────────────────────
all_combined = all_s2 + all_s3

print(f'\nTotal trials — S2: {len(all_s2)}, S3: {len(all_s3)}, Combined: {len(all_combined)}')

## Cell 6 — Trial Inventory: Which Trials Have Stim ON vs NO-STIM

In [ ]:
def print_trial_inventory(trials, sess_label):
    no_stim = [t for t in trials if not t['stim_present']]
    stim_on = [t for t in trials if     t['stim_present']]

    print(f'{"="*70}')
    print(f'  {sess_label}  —  Stim threshold: >= {STIM_THRESHOLD} mA anywhere in trial')
    print(f'{"="*70}')

    print(f'\n  NO-STIM trials (peak mA < {STIM_THRESHOLD} throughout): '
          f'n = {len(no_stim)}')
    if no_stim:
        header = f'  {"Trial":>6}  {"Digits":>6}  {"Acc":>4}  {"Presented":>10}  '\
                 f'{"Response":>10}  {"PeakMa":>8}  {"Fix_mA":>7}  {"Stim_mA":>8}  '\
                 f'{"Choice_mA":>10}  {"Fb_mA":>7}'
        print(header)
        print('  ' + '-'*100)
        for t in no_stim:
            acc_str = 'CORRECT' if t['acc']==1 else 'WRONG'
            print(f'  T{t["trial_num"]:>5}  {t["digits"]:>6}  {acc_str:>7}  '
                  f'{t["presented"]:>10}  {t["resp"]:>10}  '
                  f'{t["whole_peak_mA"]:>8.3f}  {t["fix_peak_mA"]:>7.3f}  '
                  f'{t["stim_peak_mA"]:>8.3f}  {t["ch_peak_mA"]:>10.3f}  '
                  f'{t["fb_peak_mA"]:>7.3f}')

    print(f'\n  STIM-ON trials (peak mA >= {STIM_THRESHOLD} somewhere): '
          f'n = {len(stim_on)}')
    if stim_on:
        header = f'  {"Trial":>6}  {"Digits":>6}  {"Acc":>4}  {"Presented":>10}  '\
                 f'{"Response":>10}  {"PeakMa":>8}  {"Fix_mA":>7}  {"Stim_mA":>8}  '\
                 f'{"Choice_mA":>10}  {"Fb_mA":>7}'
        print(header)
        print('  ' + '-'*100)
        for t in stim_on:
            acc_str = 'CORRECT' if t['acc']==1 else 'WRONG'
            print(f'  T{t["trial_num"]:>5}  {t["digits"]:>6}  {acc_str:>7}  '
                  f'{t["presented"]:>10}  {t["resp"]:>10}  '
                  f'{t["whole_peak_mA"]:>8.3f}  {t["fix_peak_mA"]:>7.3f}  '
                  f'{t["stim_peak_mA"]:>8.3f}  {t["ch_peak_mA"]:>10.3f}  '
                  f'{t["fb_peak_mA"]:>7.3f}')
    print()


print_trial_inventory(all_s2, 'Session 2')
print_trial_inventory(all_s3, 'Session 3')
print_trial_inventory(all_combined, 'Combined (S2 + S3)')

## Cell 7 — Statistical Comparison: NO-STIM vs STIM-ON Accuracy

In [ ]:
def print_stats(result):
    p  = result['p_value']
    d  = result['cohens_d']
    p_s = f'{p:.4f}' if not np.isnan(p) else 'N/A'
    d_s = f'{d:+.3f}' if not np.isnan(d) else 'N/A'
    print(f'  {"Group":<15}  {"N":>4}  {"Correct":>8}  {"Acc %":>7}')
    print(f'  {"-"*42}')
    print(f'  {"NO-STIM":<15}  {result["n_no_stim"]:>4}  '
          f'{result["correct_no"]:>8}  '
          f'{result["pct_no"]:>6.1f}%')
    print(f'  {"STIM-ON":<15}  {result["n_stim"]:>4}  '
          f'{result["correct_on"]:>8}  '
          f'{result["pct_on"]:>6.1f}%')
    print(f'  {"-"*42}')
    print(f'  Fisher p-value : {p_s}  {result["stars"]}')
    print(f'  Cohen\'s d      : {d_s}')
    print(f'  Effect size    : {result["effect_size"]}')
    print()


results = {}

for label, trials in [
    ('Session 2',          all_s2),
    ('Session 3',          all_s3),
    ('Combined (S2 + S3)', all_combined),
]:
    no_stim = [t for t in trials if not t['stim_present']]
    stim_on = [t for t in trials if     t['stim_present']]

    res = run_comparison(no_stim, stim_on, label)
    results[label] = res

    nos_trials = ', '.join(f'T{t["trial_num"]}' for t in no_stim)
    son_trials = ', '.join(f'T{t["trial_num"]}' for t in stim_on)

    print(f'{"="*60}')
    print(f'  {label}')
    print(f'{"="*60}')
    print(f'  NO-STIM trials : {nos_trials if nos_trials else "none"}')
    print(f'  STIM-ON trials : {son_trials if son_trials else "none"}')
    print()
    print_stats(res)

## Cell 8 — Main Comparison Plot: Accuracy by Stim Group

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# IDEA 1  —  Trial Count  (y-axis: number of trials out of 14)
# IDEA 2  —  Accuracy     (y-axis: proportion correct)
#
# Both ideas share the same 2 × 3 layout:
#   Row 1 = Condition A  (any stim anywhere → STIM-ON)
#   Row 2 = Condition B  (feedback-only stim → re-labelled NO-STIM)
#   Col 1 = Session 2  |  Col 2 = Session 3  |  Col 3 = Combined
# ══════════════════════════════════════════════════════════════════════════════

def _classify(trials, use_condB=False):
    key = 'stim_present_condB' if use_condB else 'stim_present'
    no_stim = [t for t in trials if not t[key]]
    stim_on = [t for t in trials if     t[key]]
    return no_stim, stim_on


def _run_stats(no_stim, stim_on, label):
    return run_comparison(no_stim, stim_on, label)


def _bar_label(n_stim, n_total, pct):
    """n_stim / n_total  (pct%)  — raw count and percentage inside bar label."""
    return f'{n_stim}/{n_total}\n({pct:.0f}%)'


def _stat_box(ax, res, y_pos, fontsize=9):
    p_v = res['p_value']; d_v = res['cohens_d']
    eff = effect_label(d_v); sig = sig_stars(p_v)
    p_str = f'p={p_v:.3f}' if not np.isnan(p_v) else 'p=N/A'
    d_str = f'd={d_v:+.2f}' if not np.isnan(d_v) else 'd=N/A'
    notable = (not np.isnan(p_v) and p_v < 0.05) or (not np.isnan(d_v) and abs(d_v) >= 0.8)
    fc = '#FFF0F0' if notable else '#F5F5F5'
    ec = '#C62828' if notable else '#888888'
    txt = f'{sig}  {p_str}  {d_str}  [{eff}]'
    ax.text(0.5, y_pos, txt,
            transform=ax.transAxes, ha='center', va='center',
            fontsize=fontsize, fontweight='bold', color=ec,
            bbox=dict(boxstyle='round,pad=0.35', fc=fc, ec=ec, lw=1.2, alpha=0.97))


# ─── IDEA 1 — TRIAL COUNT ─────────────────────────────────────────────────────
def plot_idea1_trial_count(all_s2, all_s3, all_combined, save_dir):
    sessions = [
        ('Session 2',          all_s2,       14),
        ('Session 3',          all_s3,       14),
        ('Combined (S2+S3)',   all_combined, 28),
    ]
    cond_labels = [
        ('A', 'Condition A: Any stim in trial → STIM-ON', False),
        ('B', 'Condition B: Feedback-only stim → NO-STIM', True),
    ]

    fig, axes = plt.subplots(2, 3, figsize=(15, 9), facecolor='white')
    fig.subplots_adjust(hspace=0.6, wspace=0.38, top=0.82, bottom=0.07,
                        left=0.07, right=0.97)

    for row, (cond_id, cond_desc, use_condB) in enumerate(cond_labels):
        for col, (sess_label, trials, n_total) in enumerate(sessions):
            ax = axes[row, col]
            no_stim, stim_on = _classify(trials, use_condB)
            res = _run_stats(no_stim, stim_on, sess_label)

            n_no = len(no_stim)
            n_on = len(stim_on)
            pct_no = 100 * n_no / n_total if n_total else 0
            pct_on = 100 * n_on / n_total if n_total else 0

            # Bars
            bars = ax.bar([0, 1], [n_no, n_on], width=0.5,
                          color=[C_NO_STIM, C_STIM],
                          edgecolor='white', linewidth=1.4, zorder=3)

            # Labels INSIDE bars
            for xi, n_val, n_tot, pct in [
                (0, n_no, n_total, pct_no),
                (1, n_on, n_total, pct_on),
            ]:
                if n_val > 0:
                    ax.text(xi, n_val / 2,
                            f'{n_val}/{n_tot}\n({pct:.0f}%)',
                            ha='center', va='center',
                            fontsize=10, fontweight='bold', color='black', zorder=8)

            # Axes
            ax.set_xlim(-0.55, 1.55)
            ax.set_ylim(0, n_total + 3)
            ax.set_yticks(range(0, n_total + 1, 2))
            ax.set_xticks([0, 1])
            ax.set_xticklabels(['NO-STIM', 'STIM-ON'], fontsize=10, fontweight='bold')
            ax.set_ylabel('Number of Trials', fontsize=9)
            ax.yaxis.grid(True, color='#eeeeee', zorder=0)
            ax.set_axisbelow(True)
            ax.set_facecolor('#FAFAFA')

            # Column title (top row only)
            if row == 0:
                ax.set_title(sess_label, fontsize=11, fontweight='bold',
                             color='#1A1A2E', pad=8)

            # Row label (left col only)
            if col == 0:
                ax.set_ylabel(f'Cond {cond_id}\nNumber of Trials', fontsize=9)

            # Stats box above bar area
            _stat_box(ax, res, 1.14, fontsize=9)

    # Suptitle
    fig.suptitle('Idea 1 — Trial Count: NO-STIM vs STIM-ON',
                 fontsize=13, fontweight='bold', color='#111', y=0.97)
    fig.text(0.5, 0.89,
             'Cond A: any stim ≥ 2 mA anywhere in trial → STIM-ON  |  '             'Cond B: feedback-only stim re-labelled → NO-STIM  |  '             'Out of 14 trials per session',
             ha='center', fontsize=8.5, color='#555')

    # Legend
    import matplotlib.patches as mpatches
    leg = [mpatches.Patch(facecolor=C_NO_STIM, label='NO-STIM'),
           mpatches.Patch(facecolor=C_STIM,    label='STIM-ON')]
    fig.legend(handles=leg, loc='lower center', ncol=2,
               fontsize=10, framealpha=0.97, facecolor='white',
               edgecolor='#ccc', bbox_to_anchor=(0.5, 0.01))

    fname = save_dir / 'idea1_trial_count.png'
    fig.savefig(fname, bbox_inches='tight', dpi=160, facecolor='white')
    plt.show(); plt.close(fig)
    print(f'Saved -> {fname}')


# ─── IDEA 2 — ACCURACY ────────────────────────────────────────────────────────
def plot_idea2_accuracy(all_s2, all_s3, all_combined, save_dir):
    sessions = [
        ('Session 2',         all_s2),
        ('Session 3',         all_s3),
        ('Combined (S2+S3)',  all_combined),
    ]
    cond_labels = [
        ('A', False),
        ('B', True),
    ]

    fig, axes = plt.subplots(2, 3, figsize=(15, 9), facecolor='white')
    fig.subplots_adjust(hspace=0.6, wspace=0.38, top=0.82, bottom=0.07,
                        left=0.07, right=0.97)

    for row, (cond_id, use_condB) in enumerate(cond_labels):
        for col, (sess_label, trials) in enumerate(sessions):
            ax = axes[row, col]
            no_stim, stim_on = _classify(trials, use_condB)
            res = _run_stats(no_stim, stim_on, sess_label)

            pct_no = res['pct_no'] if not np.isnan(res['pct_no']) else 0.0
            pct_on = res['pct_on'] if not np.isnan(res['pct_on']) else 0.0

            # Bars
            ax.bar([0, 1], [pct_no, pct_on], width=0.5,
                   color=[C_NO_STIM, C_STIM],
                   edgecolor='white', linewidth=1.4, zorder=3)

            # Labels INSIDE bars
            for xi, pct, n_corr, n_tot in [
                (0, pct_no, res['correct_no'], res['n_no_stim']),
                (1, pct_on, res['correct_on'], res['n_stim']),
            ]:
                if n_tot > 0:
                    ax.text(xi, max(pct / 2, 4),
                            f'{n_corr}/{n_tot}\n({pct:.0f}%)',
                            ha='center', va='center',
                            fontsize=10, fontweight='bold', color='black', zorder=8)

            # Chance line
            ax.axhline(50, color='#cccccc', lw=1.0, ls='--', zorder=1)

            # Axes
            ax.set_xlim(-0.55, 1.55)
            ax.set_ylim(0, 120)
            ax.set_yticks([0, 25, 50, 75, 100])
            ax.set_yticklabels(['0%', '25%', '50%', '75%', '100%'], fontsize=9)
            ax.set_xticks([0, 1])
            ax.set_xticklabels(['NO-STIM', 'STIM-ON'], fontsize=10, fontweight='bold')
            ax.yaxis.grid(True, color='#eeeeee', zorder=0)
            ax.set_axisbelow(True)
            ax.set_facecolor('#FAFAFA')

            if row == 0:
                ax.set_title(sess_label, fontsize=11, fontweight='bold',
                             color='#1A1A2E', pad=8)

            if col == 0:
                ax.set_ylabel(f'Cond {cond_id}\nAccuracy (% Correct)', fontsize=9)
            else:
                ax.set_ylabel('Accuracy (% Correct)', fontsize=9)

            _stat_box(ax, res, 1.14, fontsize=9)

    fig.suptitle('Idea 2 — Accuracy: NO-STIM vs STIM-ON',
                 fontsize=13, fontweight='bold', color='#111', y=0.97)
    fig.text(0.5, 0.89,
             'Cond A: any stim ≥ 2 mA anywhere in trial → STIM-ON  |  '             'Cond B: feedback-only stim re-labelled → NO-STIM  |  Dashed = 50% chance',
             ha='center', fontsize=8.5, color='#555')

    import matplotlib.patches as mpatches
    leg = [mpatches.Patch(facecolor=C_NO_STIM, label='NO-STIM'),
           mpatches.Patch(facecolor=C_STIM,    label='STIM-ON')]
    fig.legend(handles=leg, loc='lower center', ncol=2,
               fontsize=10, framealpha=0.97, facecolor='white',
               edgecolor='#ccc', bbox_to_anchor=(0.5, 0.01))

    fname = save_dir / 'idea2_accuracy.png'
    fig.savefig(fname, bbox_inches='tight', dpi=160, facecolor='white')
    plt.show(); plt.close(fig)
    print(f'Saved -> {fname}')


# ── Run both plots ────────────────────────────────────────────────────────────
plot_idea1_trial_count(all_s2, all_s3, all_combined, COMBINED_DIR)
plot_idea2_accuracy(all_s2, all_s3, all_combined, COMBINED_DIR)


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# IDEA 3  —  Easy = {2,3} digits  |  Hard = {4,5} digits
#            Condition B classification throughout
# ══════════════════════════════════════════════════════════════════════════════

import textwrap, matplotlib.patches as mpatches

C_EASY_NO = '#90A4AE'
C_EASY_ON = '#1A56DB'
C_HARD_NO = '#FFAB40'
C_HARD_ON = '#C62828'
C_S2      = '#7B1FA2'   # purple  — Session 2
C_S3      = '#00838F'   # teal    — Session 3


def dg(trials, easy_set, hard_set):
    """Split trials into easy/hard × no-stim/stim-on (Condition B)."""
    key = 'stim_present_condB'
    return dict(
        easy_no = [t for t in trials if t['digits'] in easy_set and not t[key]],
        easy_on = [t for t in trials if t['digits'] in easy_set and     t[key]],
        hard_no = [t for t in trials if t['digits'] in hard_set and not t[key]],
        hard_on = [t for t in trials if t['digits'] in hard_set and     t[key]],
    )


def _sbox(ax, res, y_ax=1.09, fs=8.5):
    p_v = res['p_value']; d_v = res['cohens_d']
    eff = effect_label(d_v); sig = sig_stars(p_v)
    p_str = f'p={p_v:.3f}' if not np.isnan(p_v) else 'p=N/A'
    d_str = f'd={d_v:+.2f}' if not np.isnan(d_v) else 'd=N/A'
    notable = (not np.isnan(p_v) and p_v < 0.05) or \
              (not np.isnan(d_v) and abs(d_v) >= 0.8)
    fc = '#FFF0F0' if notable else '#F5F5F5'
    ec = '#C62828' if notable else '#888888'
    ax.text(0.5, y_ax, f'{sig}  {p_str}  {d_str}  [{eff}]',
            transform=ax.transAxes, ha='center', va='center',
            fontsize=fs, fontweight='bold', color=ec,
            bbox=dict(boxstyle='round,pad=0.3', fc=fc, ec=ec, lw=1.1, alpha=0.97))


def _bar2(ax, g1, g2, c1, c2, xl, show_stats, n_total=None, y_type='count'):
    """Draw a clean 2-bar subplot. y_type='count' or 'acc'."""
    if y_type == 'count':
        v1, v2 = len(g1), len(g2)
        ax.bar([0,1], [v1,v2], width=0.5, color=[c1,c2],
               edgecolor='white', linewidth=1.4, zorder=3)
        n_tot = n_total or max(v1+v2, 1)
        for xi, nv in [(0,v1),(1,v2)]:
            pct = 100*nv/n_tot if n_tot else 0
            if nv > 0:
                ax.text(xi, nv/2, f'{nv}\n({pct:.0f}%)',
                        ha='center', va='center',
                        fontsize=9.5, fontweight='bold', color='white', zorder=8)
        max_y = max(v1, v2, 1) + 2
        ax.set_ylim(0, max_y)
        ax.set_yticks(range(0, int(max_y)+1, max(1, int(max_y)//4)))
        ax.set_ylabel('# Trials', fontsize=8)
    else:
        res = run_comparison(g1, g2, '')
        p1 = res['pct_no'] if not np.isnan(res['pct_no']) else 0.0
        p2 = res['pct_on'] if not np.isnan(res['pct_on']) else 0.0
        ax.bar([0,1], [p1,p2], width=0.5, color=[c1,c2],
               edgecolor='white', linewidth=1.4, zorder=3)
        for xi, pct, nc, nt in [(0,p1,res['correct_no'],res['n_no_stim']),
                                  (1,p2,res['correct_on'],res['n_stim'])]:
            if nt > 0:
                ax.text(xi, max(pct/2,5), f'{nc}/{nt}\n({pct:.0f}%)',
                        ha='center', va='center',
                        fontsize=9.5, fontweight='bold', color='white', zorder=8)
        ax.axhline(50, color='#cccccc', lw=1.0, ls='--', zorder=1)
        ax.set_ylim(0, 125)
        ax.set_yticks([0,25,50,75,100])
        ax.set_yticklabels(['0%','25%','50%','75%','100%'], fontsize=8)
        ax.set_ylabel('Accuracy (%)', fontsize=8)
        if show_stats:
            _sbox(ax, res, y_ax=1.10, fs=8)

    ax.set_xticks([0,1])
    ax.set_xticklabels(xl, fontsize=9, fontweight='bold')
    ax.set_xlim(-0.55, 1.55)
    ax.yaxis.grid(True, color='#eeeeee', zorder=0)
    ax.set_axisbelow(True)
    ax.set_facecolor('#FAFAFA')
    return run_comparison(g1, g2, '') if y_type=='acc' else None


def _fig1x3(title, subtitle, col_titles, legend_handles, tight_top=0.82):
    fig, axes = plt.subplots(1, 3, figsize=(13, 5.5), facecolor='white')
    fig.subplots_adjust(wspace=0.40, top=tight_top, bottom=0.13,
                        left=0.07, right=0.97)
    fig.suptitle(title, fontsize=12, fontweight='bold', color='#111', y=0.97)
    fig.text(0.5, 0.92, subtitle, ha='center', fontsize=8.5, color='#555')
    for ax, ct in zip(axes, col_titles):
        ax.set_title(ct, fontsize=11, fontweight='bold', color='#1A1A2E', pad=8)
    if legend_handles:
        fig.legend(handles=legend_handles, loc='lower center', ncol=len(legend_handles),
                   fontsize=9, framealpha=0.97, facecolor='white',
                   edgecolor='#ccc', bbox_to_anchor=(0.5, 0.01))
    return fig, axes


def save_fig(fig, path):
    fig.savefig(path, bbox_inches='tight', dpi=160, facecolor='white')
    plt.show(); plt.close(fig)
    print(f'Saved -> {path}')


# ─────────────────────────────────────────────────────────────────────────────
def run_idea(easy_set, hard_set, idea_tag, save_dir):
    easy_str = '+'.join(str(d) for d in sorted(easy_set))
    hard_str = '+'.join(str(d) for d in sorted(hard_set))
    tag_str  = f'Easy={easy_str}d  Hard={hard_str}d'
    it = idea_tag  # short

    s2g = dg(all_s2,       easy_set, hard_set)
    s3g = dg(all_s3,       easy_set, hard_set)
    cg  = dg(all_combined, easy_set, hard_set)

    # Total within each session for percentage denominator
    n_s2 = len([t for t in all_s2 if t['digits'] in easy_set|hard_set])
    n_s3 = len([t for t in all_s3 if t['digits'] in easy_set|hard_set])
    n_cb = n_s2 + n_s3

    leg_stim = [mpatches.Patch(facecolor=C_EASY_NO, label='NO-STIM'),
                mpatches.Patch(facecolor=C_EASY_ON, label='STIM-ON')]
    leg_hard = [mpatches.Patch(facecolor=C_HARD_NO, label='NO-STIM'),
                mpatches.Patch(facecolor=C_HARD_ON, label='STIM-ON')]
    leg_eh   = [mpatches.Patch(facecolor='#5C6BC0', label='Easy'),
                mpatches.Patch(facecolor='#EF6C00', label='Hard')]
    leg_sess = [mpatches.Patch(facecolor=C_S2, label='Session 2'),
                mpatches.Patch(facecolor=C_S3, label='Session 3')]

    # ── PLOT A — Easy: S2 | S3 | Combined  (trial count, no stats) ──────────
    fig, axes = _fig1x3(
        f'{it} Plot A — Easy Trials: NO-STIM vs STIM-ON  (Trial Count)',
        f'{tag_str}  |  Condition B  |  No statistics',
        ['Session 2', 'Session 3', 'Combined'],
        leg_stim)
    for ax, grp, nt in zip(axes, [s2g,s3g,cg], [n_s2,n_s3,n_cb]):
        _bar2(ax, grp['easy_no'], grp['easy_on'],
              C_EASY_NO, C_EASY_ON, ['NO-STIM','STIM-ON'],
              show_stats=False, n_total=nt, y_type='count')
    save_fig(fig, save_dir / f'{it.lower().replace(" ","_")}_A_easy_count.png')

    # ── PLOT B — Hard: S2 | S3 | Combined  (trial count, no stats) ──────────
    fig, axes = _fig1x3(
        f'{it} Plot B — Hard Trials: NO-STIM vs STIM-ON  (Trial Count)',
        f'{tag_str}  |  Condition B  |  No statistics',
        ['Session 2', 'Session 3', 'Combined'],
        leg_hard)
    for ax, grp, nt in zip(axes, [s2g,s3g,cg], [n_s2,n_s3,n_cb]):
        _bar2(ax, grp['hard_no'], grp['hard_on'],
              C_HARD_NO, C_HARD_ON, ['NO-STIM','STIM-ON'],
              show_stats=False, n_total=nt, y_type='count')
    save_fig(fig, save_dir / f'{it.lower().replace(" ","_")}_B_hard_count.png')

    # ── PLOT C — Easy vs Hard: S2 | S3 | Combined  (trial count, no stats) ──
    fig, axes = _fig1x3(
        f'{it} Plot C — Easy vs Hard  (Trial Count)',
        f'{tag_str}  |  Condition B  |  No statistics',
        ['Session 2', 'Session 3', 'Combined'],
        leg_eh)
    for ax, grp, nt in zip(axes, [s2g,s3g,cg], [n_s2,n_s3,n_cb]):
        easy_all = grp['easy_no'] + grp['easy_on']
        hard_all = grp['hard_no'] + grp['hard_on']
        _bar2(ax, easy_all, hard_all,
              '#5C6BC0', '#EF6C00', ['Easy','Hard'],
              show_stats=False, n_total=nt, y_type='count')
    save_fig(fig, save_dir / f'{it.lower().replace(" ","_")}_C_easy_vs_hard_count.png')

    # ── PLOT D — Easy: S2 | S3 | Combined  (accuracy, with stats) ───────────
    fig, axes = _fig1x3(
        f'{it} Plot D — Easy Trials: NO-STIM vs STIM-ON  (Accuracy)',
        f'{tag_str}  |  Condition B  |  Fisher p + Cohen d + effect',
        ['Session 2', 'Session 3', 'Combined'],
        leg_stim, tight_top=0.80)
    for ax, grp in zip(axes, [s2g,s3g,cg]):
        _bar2(ax, grp['easy_no'], grp['easy_on'],
              C_EASY_NO, C_EASY_ON, ['NO-STIM','STIM-ON'],
              show_stats=True, y_type='acc')
    save_fig(fig, save_dir / f'{it.lower().replace(" ","_")}_D_easy_acc.png')

    # ── PLOT E — Hard: S2 | S3 | Combined  (accuracy, with stats) ───────────
    fig, axes = _fig1x3(
        f'{it} Plot E — Hard Trials: NO-STIM vs STIM-ON  (Accuracy)',
        f'{tag_str}  |  Condition B  |  Fisher p + Cohen d + effect',
        ['Session 2', 'Session 3', 'Combined'],
        leg_hard, tight_top=0.80)
    for ax, grp in zip(axes, [s2g,s3g,cg]):
        _bar2(ax, grp['hard_no'], grp['hard_on'],
              C_HARD_NO, C_HARD_ON, ['NO-STIM','STIM-ON'],
              show_stats=True, y_type='acc')
    save_fig(fig, save_dir / f'{it.lower().replace(" ","_")}_E_hard_acc.png')

    # ── PLOT F — Easy vs Hard: S2 | S3 | Combined  (accuracy, with stats) ───
    fig, axes = _fig1x3(
        f'{it} Plot F — Easy vs Hard  (Accuracy)',
        f'{tag_str}  |  Condition B  |  Fisher p + Cohen d + effect',
        ['Session 2', 'Session 3', 'Combined'],
        leg_eh, tight_top=0.80)
    for ax, grp in zip(axes, [s2g,s3g,cg]):
        easy_all = grp['easy_no'] + grp['easy_on']
        hard_all = grp['hard_no'] + grp['hard_on']
        _bar2(ax, easy_all, hard_all,
              '#5C6BC0', '#EF6C00', ['Easy','Hard'],
              show_stats=True, y_type='acc')
    save_fig(fig, save_dir / f'{it.lower().replace(" ","_")}_F_easy_vs_hard_acc.png')

    # ══ BETWEEN-SESSION COMPARISONS ════════════════════════════════════════

    # ── PLOT G — Easy NO-STIM vs STIM-ON: S2 Easy | S3 Easy | S2-Easy vs S3-Easy  (count)
    fig, axes = _fig1x3(
        f'{it} Plot G — Easy: Between-Session  (Trial Count)',
        f'{tag_str}  |  Condition B  |  No statistics',
        ['Session 2  Easy', 'Session 3  Easy', 'S2-Easy vs S3-Easy'],
        [mpatches.Patch(facecolor=C_EASY_NO, label='NO-STIM (S2/S3)'),
         mpatches.Patch(facecolor=C_EASY_ON, label='STIM-ON (S2/S3)'),
         mpatches.Patch(facecolor=C_S2, label='S2 Easy (all)'),
         mpatches.Patch(facecolor=C_S3, label='S3 Easy (all)')])
    _bar2(axes[0], s2g['easy_no'], s2g['easy_on'],
          C_EASY_NO, C_EASY_ON, ['NO-STIM','STIM-ON'],
          show_stats=False, n_total=n_s2, y_type='count')
    _bar2(axes[1], s3g['easy_no'], s3g['easy_on'],
          C_EASY_NO, C_EASY_ON, ['NO-STIM','STIM-ON'],
          show_stats=False, n_total=n_s3, y_type='count')
    s2_easy_all = s2g['easy_no'] + s2g['easy_on']
    s3_easy_all = s3g['easy_no'] + s3g['easy_on']
    _bar2(axes[2], s2_easy_all, s3_easy_all,
          C_S2, C_S3, ['S2 Easy','S3 Easy'],
          show_stats=False, n_total=n_s2+n_s3, y_type='count')
    save_fig(fig, save_dir / f'{it.lower().replace(" ","_")}_G_easy_between_count.png')

    # ── PLOT H — Hard NO-STIM vs STIM-ON between sessions  (count) ──────────
    fig, axes = _fig1x3(
        f'{it} Plot H — Hard: Between-Session  (Trial Count)',
        f'{tag_str}  |  Condition B  |  No statistics',
        ['Session 2  Hard', 'Session 3  Hard', 'S2-Hard vs S3-Hard'],
        [mpatches.Patch(facecolor=C_HARD_NO, label='NO-STIM (S2/S3)'),
         mpatches.Patch(facecolor=C_HARD_ON, label='STIM-ON (S2/S3)'),
         mpatches.Patch(facecolor=C_S2, label='S2 Hard (all)'),
         mpatches.Patch(facecolor=C_S3, label='S3 Hard (all)')])
    _bar2(axes[0], s2g['hard_no'], s2g['hard_on'],
          C_HARD_NO, C_HARD_ON, ['NO-STIM','STIM-ON'],
          show_stats=False, n_total=n_s2, y_type='count')
    _bar2(axes[1], s3g['hard_no'], s3g['hard_on'],
          C_HARD_NO, C_HARD_ON, ['NO-STIM','STIM-ON'],
          show_stats=False, n_total=n_s3, y_type='count')
    s2_hard_all = s2g['hard_no'] + s2g['hard_on']
    s3_hard_all = s3g['hard_no'] + s3g['hard_on']
    _bar2(axes[2], s2_hard_all, s3_hard_all,
          C_S2, C_S3, ['S2 Hard','S3 Hard'],
          show_stats=False, n_total=n_s2+n_s3, y_type='count')
    save_fig(fig, save_dir / f'{it.lower().replace(" ","_")}_H_hard_between_count.png')

    # ── PLOT I — Easy vs Hard between sessions  (count) ──────────────────────
    fig, axes = _fig1x3(
        f'{it} Plot I — Easy vs Hard: Between-Session  (Trial Count)',
        f'{tag_str}  |  Condition B  |  No statistics',
        ['Session 2', 'Session 3', 'S2 vs S3 (Easy+Hard)'],
        [mpatches.Patch(facecolor='#5C6BC0', label='Easy'),
         mpatches.Patch(facecolor='#EF6C00', label='Hard'),
         mpatches.Patch(facecolor=C_S2, label='Session 2 (all)'),
         mpatches.Patch(facecolor=C_S3, label='Session 3 (all)')])
    for ax, grp, nt in zip(axes[:2], [s2g,s3g], [n_s2,n_s3]):
        _bar2(ax, grp['easy_no']+grp['easy_on'],
              grp['hard_no']+grp['hard_on'],
              '#5C6BC0', '#EF6C00', ['Easy','Hard'],
              show_stats=False, n_total=nt, y_type='count')
    all_s2_diff = s2g['easy_no']+s2g['easy_on']+s2g['hard_no']+s2g['hard_on']
    all_s3_diff = s3g['easy_no']+s3g['easy_on']+s3g['hard_no']+s3g['hard_on']
    _bar2(axes[2], all_s2_diff, all_s3_diff,
          C_S2, C_S3, ['Session 2','Session 3'],
          show_stats=False, n_total=n_s2+n_s3, y_type='count')
    save_fig(fig, save_dir / f'{it.lower().replace(" ","_")}_I_between_easy_hard_count.png')

    # ── PLOT G2 — Easy between-session  (accuracy, with stats) ──────────────
    fig, axes = _fig1x3(
        f'{it} Plot G2 — Easy: Between-Session  (Accuracy)',
        f'{tag_str}  |  Condition B  |  Fisher p + Cohen d + effect',
        ['Session 2  Easy', 'Session 3  Easy', 'S2-Easy vs S3-Easy'],
        [mpatches.Patch(facecolor=C_EASY_NO, label='NO-STIM'),
         mpatches.Patch(facecolor=C_EASY_ON, label='STIM-ON'),
         mpatches.Patch(facecolor=C_S2, label='S2 Easy (all)'),
         mpatches.Patch(facecolor=C_S3, label='S3 Easy (all)')],
        tight_top=0.80)
    _bar2(axes[0], s2g['easy_no'], s2g['easy_on'],
          C_EASY_NO, C_EASY_ON, ['NO-STIM','STIM-ON'],
          show_stats=True, y_type='acc')
    _bar2(axes[1], s3g['easy_no'], s3g['easy_on'],
          C_EASY_NO, C_EASY_ON, ['NO-STIM','STIM-ON'],
          show_stats=True, y_type='acc')
    _bar2(axes[2], s2_easy_all, s3_easy_all,
          C_S2, C_S3, ['S2 Easy','S3 Easy'],
          show_stats=True, y_type='acc')
    save_fig(fig, save_dir / f'{it.lower().replace(" ","_")}_G2_easy_between_acc.png')

    # ── PLOT H2 — Hard between-session  (accuracy, with stats) ──────────────
    fig, axes = _fig1x3(
        f'{it} Plot H2 — Hard: Between-Session  (Accuracy)',
        f'{tag_str}  |  Condition B  |  Fisher p + Cohen d + effect',
        ['Session 2  Hard', 'Session 3  Hard', 'S2-Hard vs S3-Hard'],
        [mpatches.Patch(facecolor=C_HARD_NO, label='NO-STIM'),
         mpatches.Patch(facecolor=C_HARD_ON, label='STIM-ON'),
         mpatches.Patch(facecolor=C_S2, label='S2 Hard (all)'),
         mpatches.Patch(facecolor=C_S3, label='S3 Hard (all)')],
        tight_top=0.80)
    _bar2(axes[0], s2g['hard_no'], s2g['hard_on'],
          C_HARD_NO, C_HARD_ON, ['NO-STIM','STIM-ON'],
          show_stats=True, y_type='acc')
    _bar2(axes[1], s3g['hard_no'], s3g['hard_on'],
          C_HARD_NO, C_HARD_ON, ['NO-STIM','STIM-ON'],
          show_stats=True, y_type='acc')
    _bar2(axes[2], s2_hard_all, s3_hard_all,
          C_S2, C_S3, ['S2 Hard','S3 Hard'],
          show_stats=True, y_type='acc')
    save_fig(fig, save_dir / f'{it.lower().replace(" ","_")}_H2_hard_between_acc.png')

    # ── PLOT I2 — Easy vs Hard between sessions  (accuracy, with stats) ──────
    fig, axes = _fig1x3(
        f'{it} Plot I2 — Easy vs Hard: Between-Session  (Accuracy)',
        f'{tag_str}  |  Condition B  |  Fisher p + Cohen d + effect',
        ['Session 2', 'Session 3', 'S2 vs S3 (Easy+Hard)'],
        [mpatches.Patch(facecolor='#5C6BC0', label='Easy'),
         mpatches.Patch(facecolor='#EF6C00', label='Hard'),
         mpatches.Patch(facecolor=C_S2, label='Session 2 (all)'),
         mpatches.Patch(facecolor=C_S3, label='Session 3 (all)')],
        tight_top=0.80)
    for ax, grp in zip(axes[:2], [s2g,s3g]):
        _bar2(ax, grp['easy_no']+grp['easy_on'],
              grp['hard_no']+grp['hard_on'],
              '#5C6BC0', '#EF6C00', ['Easy','Hard'],
              show_stats=True, y_type='acc')
    _bar2(axes[2], all_s2_diff, all_s3_diff,
          C_S2, C_S3, ['Session 2','Session 3'],
          show_stats=True, y_type='acc')
    save_fig(fig, save_dir / f'{it.lower().replace(" ","_")}_I2_between_easy_hard_acc.png')

    # ── PLOT J — Trial Membership Overview ───────────────────────────────────
    excl_col = '#E0E0E0'
    col_map = {('easy',False):C_EASY_NO, ('easy',True):C_EASY_ON,
               ('hard',False):C_HARD_NO, ('hard',True):C_HARD_ON}

    fig, axes = plt.subplots(1, 2, figsize=(16, 5), facecolor='white')
    fig.subplots_adjust(wspace=0.30, top=0.78, bottom=0.16, left=0.07, right=0.97)

    for ax, (sess_label, trials) in zip(axes, [('Session 2', all_s2),
                                                 ('Session 3', all_s3)]):
        lookup = {}
        for t in trials:
            diff = 'easy' if t['digits'] in easy_set else \
                   'hard' if t['digits'] in hard_set else 'excl'
            lookup[t['trial_num']] = (diff, t['stim_present_condB'], t['digits'], t['acc'])

        for ri, diff in enumerate(['easy','hard']):
            for tn in range(1, 15):
                x0, y0 = tn-1, ri
                if tn not in lookup or lookup[tn][0] != diff:
                    fc, txt, tc = excl_col, f'T{tn}\n—', '#aaa'
                else:
                    _, stim_on, nd, acc = lookup[tn]
                    fc = col_map[(diff, stim_on)]
                    txt = f'T{tn}\n{nd}d {"✓" if acc==1 else "✗"}'
                    tc = 'white'
                ax.add_patch(plt.Rectangle((x0+0.05, y0+0.05), 0.90, 0.90,
                                            facecolor=fc, edgecolor='white', linewidth=1.5))
                ax.text(x0+0.5, y0+0.5, txt, ha='center', va='center',
                        fontsize=7.5, color=tc, fontweight='bold', linespacing=1.3)

        ax.set_xlim(0, 14); ax.set_ylim(0, 2)
        ax.set_xticks([]); ax.set_yticks([0.5, 1.5])
        ax.set_yticklabels([
            f'Easy ({easy_str}d)', f'Hard ({hard_str}d)'
        ], fontsize=9)
        ax.set_title(sess_label, fontsize=11, fontweight='bold', color='#1A1A2E', pad=8)
        ax.set_facecolor('white')
        for sp in ax.spines.values(): sp.set_visible(False)
        for tn in range(1, 15):
            ax.text(tn-0.5, -0.12, f'T{tn}', ha='center', va='top',
                    fontsize=7, color='#555')

    fig.suptitle(f'{it} Plot J — Trial Membership  ({tag_str})',
                 fontsize=12, fontweight='bold', color='#111', y=0.97)
    fig.text(0.5, 0.90, 'Colour = difficulty × stim group (Cond B)  |  '
             'Cell: T# | digit-length | ✓ correct / ✗ incorrect',
             ha='center', fontsize=8, color='#555')
    fig.legend(handles=[
        mpatches.Patch(facecolor=C_EASY_NO, label='Easy  NO-STIM'),
        mpatches.Patch(facecolor=C_EASY_ON, label='Easy  STIM-ON'),
        mpatches.Patch(facecolor=C_HARD_NO, label='Hard  NO-STIM'),
        mpatches.Patch(facecolor=C_HARD_ON, label='Hard  STIM-ON'),
        mpatches.Patch(facecolor=excl_col,  label='Not in category'),
    ], loc='lower center', ncol=5, fontsize=9, framealpha=0.97,
       facecolor='white', edgecolor='#ccc', bbox_to_anchor=(0.5, 0.01))
    save_fig(fig, save_dir / f'{it.lower().replace(" ","_")}_J_membership.png')


# ── RUN ───────────────────────────────────────────────────────────────────────
run_idea({2,3}, {4,5}, 'Idea 3', COMBINED_DIR)


## Idea 4 — Difficulty Split: Easy = 2 digits only | Hard = 3–4–5 digits
Same plot structure as Idea 3 (Plots A–J), reusing all functions.


In [ ]:
run_idea({2}, {3,4,5}, 'Idea 4', COMBINED_DIR)


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Grouped Bar: Stim ON vs OFF × Easy/Hard × Session 2 vs Session 3
# Two cutoffs: mA >= 1 and mA >= 2  |  Condition B logic
# Requires: all_s2, all_s3, mpatches, np, plt, save_fig, COMBINED_DIR
# ══════════════════════════════════════════════════════════════════════════════

_C_ON   = '#ef4444'
_C_OFF  = '#3b82f6'
_EASY_S = {2,3}
_HARD_S = {4, 5}

def _classify_cutoff(trials, cutoff):
    out = []
    for t in trials:
        t2 = t.copy()
        t2['_stim'] = (t['whole_peak_mA'] >= cutoff) and not t['feedback_only_stim']
        out.append(t2)
    return out

def _get_grp(trials, dset, stim_on):
    return [t for t in trials if t['digits'] in dset and t['_stim'] == stim_on]

def _plot_panel(ax, s2_trials, s3_trials, cutoff, easy_set, hard_set):
    s2 = _classify_cutoff(s2_trials, cutoff)
    s3 = _classify_cutoff(s3_trials, cutoff)

    groups_def = [
        ('S2 Easy', s2, easy_set),
        ('S2 Hard', s2, hard_set),
        ('S3 Easy', s3, easy_set),
        ('S3 Hard', s3, hard_set),
    ]

    bw  = 0.32
    gap = 0.06
    xc  = np.arange(len(groups_def))

    for gi, (glabel, trials, dset) in enumerate(groups_def):
        on_g  = _get_grp(trials, dset, True)
        off_g = _get_grp(trials, dset, False)

        x_on  = xc[gi] - gap/2 - bw
        x_off = xc[gi] + gap/2

        for x, grp, col in [(x_on, on_g, _C_ON), (x_off, off_g, _C_OFF)]:
            n = len(grp)
            ax.bar(x, n, width=bw, color=col,
                   edgecolor='white', linewidth=1.2, zorder=3)
            ax.text(x, n + 0.15, str(n),
                    ha='center', va='bottom', fontsize=10,
                    fontweight='bold', color='#111')
            if n > 0:
                c   = sum(t['acc'] for t in grp)
                pct = int(round(100 * c / n))
                lbl = f'{c}/{n}\n({pct}%)'
                if n >= 3:
                    ax.text(x, n / 2, lbl, ha='center', va='center',
                            fontsize=7.5, fontweight='600', color='white',
                            linespacing=1.3, zorder=8)
                else:
                    ax.text(x, n + 0.9, lbl, ha='center', va='bottom',
                            fontsize=7, fontweight='600', color=col,
                            linespacing=1.3)

    ax.set_xticks(xc)
    ax.set_xticklabels([g[0] for g in groups_def], fontsize=9.5, fontweight='bold')
    ax.set_ylim(0, 16)
    ax.set_yticks(range(0, 15, 2))
    ax.set_ylabel('Number of Trials', fontsize=8.5)
    ax.yaxis.grid(True, color='#eeeeee', zorder=0)
    ax.set_axisbelow(True)
    ax.set_facecolor('#FAFAFA')

    easy_str = '+'.join(str(d) for d in sorted(easy_set))
    hard_str = '+'.join(str(d) for d in sorted(hard_set))
    n_s2e = len([t for t in s2_trials if t['digits'] in easy_set])
    n_s2h = len([t for t in s2_trials if t['digits'] in hard_set])
    n_s3e = len([t for t in s3_trials if t['digits'] in easy_set])
    n_s3h = len([t for t in s3_trials if t['digits'] in hard_set])
    ax.set_title(
        f'Cut-off ≥ {cutoff} mA  |  Easy = {easy_str}d, Hard = {hard_str}d\n'
        f'S2: {n_s2e} Easy + {n_s2h} Hard  |  S3: {n_s3e} Easy + {n_s3h} Hard',
        fontsize=9, fontweight='bold', color='#111827', pad=8, loc='left'
    )

# ── Figure ────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5), facecolor='white')
fig.subplots_adjust(wspace=0.38, top=0.82, bottom=0.13, left=0.07, right=0.97)

for ax, cutoff in zip(axes, [1, 2]):
    _plot_panel(ax, all_s2, all_s3, cutoff, _EASY_S, _HARD_S)

fig.suptitle('Stim On vs. Stim OFF — Number of Trials  (Session 2 vs Session 3)',
             fontsize=13, fontweight='bold', color='#111', y=0.97)
fig.text(0.5, 0.90,
         'Session 2 & Session 3 · 14 trials each  |  '
         'Easy = 2+3d, Hard = 3+4+5d  |  Accuracy label = correct/total (%)',
         ha='center', fontsize=8.5, color='#555')
fig.legend(
    handles=[mpatches.Patch(facecolor=_C_ON,  label='Stim ON'),
             mpatches.Patch(facecolor=_C_OFF, label='Stim OFF')],
    loc='lower center', ncol=2, fontsize=10, framealpha=0.97,
    facecolor='white', edgecolor='#ccc', bbox_to_anchor=(0.5, 0.01)
)

save_fig(fig, COMBINED_DIR / 's2_vs_s3_grouped_stim_accuracy_cutoff1_2.png')